<div style="background-color: #ffffff; color: #000000; padding: 30px;">
<img src="../media/images/kisz_logo.png" width="192" height="69" align="right" style="margin-right: 50px; margin-bottom: 50px;">
<h1>Time Series Analysis and Forecasting</h1>
</div>

<div style="background-color: #f6a800; color: #ffffff; padding: 10px;">
<h2>Part C: Machine Learning Approaches</h2>
<h2>Notebook C03: Ensembles and Model Combinations</h2>
</div>

We now have four quite different forecasters: a linear model, a random forest, a gradient booster, and
SARIMAX. The obvious next step is to pick the best one. The alternative is to use all of them.

Combining forecasts is one of the oldest results in the field and one of the most reliable. It also comes
with more caveats than its reputation suggests, and this notebook is as interested in those as in the
methods. We build every standard combination, then test whether any of them actually beat simply picking
the best model.

---

**Contents**

1. [Imports and the Base Models](#1.-Imports-and-the-Base-Models)
2. [Why Combining Can Help](#2.-Why-Combining-Can-Help)
3. [Simple Ensembles: Mean and Median](#3.-Simple-Ensembles:-Mean-and-Median)
4. [Optimal Weights](#4.-Optimal-Weights)
5. [Stacking](#5.-Stacking)
6. [Does It Actually Help?](#6.-Does-It-Actually-Help?)
7. [When to Combine](#7.-When-to-Combine)

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="1.-Imports-and-the-Base-Models">1. Imports and the Base Models</h3>
</div>

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from scipy.optimize import minimize
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.arima.model import ARIMA

import nb_config

sns.set_theme(style="whitegrid")

The same features and store as Notebook [C02](./C02_Machine_learning_models.ipynb).

The split is the important new part. Combining forecasts needs its own data. Weights learned from the
predictions the models made on their *training* data would be meaningless, because every model looks good
there. So we need three periods:

| Period | Used for |
|---|---|
| **Train** | Fitting the base models |
| **Validation** | Base models predict it; the combination is learned from those predictions |
| **Test** | Judging the combination, never seen by anything |

Skipping the middle one is the standard way to produce an ensemble that looks excellent and is not.

In [ ]:
sales = pd.read_csv(nb_config.ROSSMANN_TRAIN_PATH, parse_dates=["Date"], low_memory=False)
store = sales[sales["Store"] == 1].set_index("Date").sort_index().asfreq("D")
target = store["Sales"].astype(float)


def build_features(target, store):
    """As in C01 and C02: lags, shifted rolling statistics, calendar, known-in-advance."""
    features = pd.DataFrame(index=target.index)

    for lag in (1, 2, 7, 14, 28):
        features[f"lag_{lag}"] = target.shift(lag)

    history = target.shift(1)
    for window in (7, 28):
        features[f"roll_mean_{window}"] = history.rolling(window).mean()
        features[f"roll_std_{window}"] = history.rolling(window).std()

    features["day_of_week"] = target.index.dayofweek
    features["day_of_month"] = target.index.day
    features["month"] = target.index.month
    features["days_since_start"] = (target.index - target.index[0]).days

    for k in (1, 2):
        position = target.index.dayofyear / 365.25
        features[f"fourier_sin_{k}"] = np.sin(2 * np.pi * k * position)
        features[f"fourier_cos_{k}"] = np.cos(2 * np.pi * k * position)

    features["open"] = store["Open"]
    features["promo"] = store["Promo"]
    features["school_holiday"] = store["SchoolHoliday"]

    return features


features = build_features(target, store)
complete = features.notna().all(axis=1)
X, y = features[complete], target[complete]
exogenous = store[["Open", "Promo"]].astype(float).loc[X.index]

VALIDATION_DAYS = 90
TEST_DAYS = 90

test_start = len(X) - TEST_DAYS
validation_start = test_start - VALIDATION_DAYS

print(f"Train:      {X.index[0].date()} to {X.index[validation_start - 1].date()}  "
      f"({validation_start} days)")
print(f"Validation: {X.index[validation_start].date()} to {X.index[test_start - 1].date()}  "
      f"({VALIDATION_DAYS} days)")
print(f"Test:       {X.index[test_start].date()} to {X.index[-1].date()}  ({TEST_DAYS} days)")

In [ ]:
def fit_base_models(X, y, exogenous, validation_start, test_start, horizon):
    """Fit four different forecasters and collect their validation and test predictions."""
    learners = {
        "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=10.0)),
        "Random forest": RandomForestRegressor(n_estimators=300, random_state=0, n_jobs=-1),
        "LightGBM": lgb.LGBMRegressor(
            n_estimators=200, learning_rate=0.03, num_leaves=7, random_state=0, verbose=-1
        ),
    }

    validation, test = {}, {}

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")

        for name, learner in learners.items():
            learner.fit(X.iloc[:validation_start], y.iloc[:validation_start])
            validation[name] = learner.predict(X.iloc[validation_start:test_start])
            test[name] = learner.predict(X.iloc[test_start:test_start + horizon])

        # SARIMAX is refitted for each period, since it forecasts from the end of its data
        for label, (start, end, store_in) in {
            "validation": (validation_start, test_start, validation),
            "test": (test_start, test_start + horizon, test),
        }.items():
            model = ARIMA(
                y.iloc[:start], exog=exogenous.iloc[:start],
                order=(1, 0, 1), seasonal_order=(1, 0, 1, 7),
            ).fit()
            store_in["SARIMAX"] = model.forecast(end - start, exog=exogenous.iloc[start:end]).values

    index_validation = X.index[validation_start:test_start]
    index_test = X.index[test_start:test_start + horizon]

    return (pd.DataFrame(validation, index=index_validation),
            pd.DataFrame(test, index=index_test))


validation_predictions, test_predictions = fit_base_models(
    X, y, exogenous, validation_start, test_start, TEST_DAYS
)

y_validation = y.iloc[validation_start:test_start]
y_test = y.iloc[test_start:]

individual = pd.Series(
    {name: mean_absolute_error(y_test, test_predictions[name]) for name in test_predictions}
).sort_values()

individual.round(1)

Two strong models and two weak ones, which is a realistic situation and, as we will see, an awkward one
for the simpler combination methods.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="2.-Why-Combining-Can-Help">2. Why Combining Can Help</h3>
</div>

Combining helps when models make **different mistakes**. If two forecasts are wrong in the same way at the
same time, averaging them changes nothing; if one is high whenever the other is low, the average is better
than either.

That is measurable. Correlate the models' errors on the validation period, where none of them was fitted.

In [ ]:
validation_errors = validation_predictions.sub(y_validation, axis=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5),
                         gridspec_kw={"width_ratios": [1, 1.4]})

sns.heatmap(validation_errors.corr(), annot=True, fmt=".2f", cmap="RdYlBu_r",
            vmin=0, vmax=1, square=True, ax=axes[0], cbar=False)
axes[0].set_title("Error correlation", fontsize=13, fontweight="bold")

for name in validation_errors.columns:
    axes[1].plot(validation_errors.index, validation_errors[name], linewidth=1.0, label=name)
axes[1].axhline(0, color="black", linewidth=1.0)
axes[1].set_title("Errors over the validation period", fontsize=13, fontweight="bold")
axes[1].set_ylabel("Forecast - actual")
axes[1].legend(fontsize=9)
axes[1].tick_params(axis="x", rotation=45)
axes[1].grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

The correlations run from 0.69 to 0.93, all firmly positive. The models mostly get the same days wrong,
which is unsurprising: they share a feature matrix, and the hardest days are hard for structural reasons
that no model can see.

The highest pair is the random forest and LightGBM at 0.93. Two tree ensembles on identical features are
close to the same model. The lowest is the random forest against Ridge at 0.69, which is the pairing with
the most genuinely different view.

This already sets expectations. Combining works best with **diverse and comparably good** models. Ours are
correlated, and two of them are much worse than the others.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="3.-Simple-Ensembles:-Mean-and-Median">3. Simple Ensembles: Mean and Median</h3>
</div>

The simplest combination is an unweighted **mean**. It needs no fitting, has no parameters, and cannot
overfit, which is why it is a surprisingly hard benchmark in forecasting competitions.

The **median** is its robust cousin: it ignores a single model that goes badly wrong on a given day, which
matters when one model occasionally produces something absurd.

In [ ]:
combinations = pd.DataFrame(index=test_predictions.index)
combinations["Mean"] = test_predictions.mean(axis=1)
combinations["Median"] = test_predictions.median(axis=1)

simple_scores = pd.Series({
    name: mean_absolute_error(y_test, combinations[name]) for name in combinations
})

print("Individual models:")
print(individual.round(1).to_string())
print()
print("Simple combinations:")
print(simple_scores.round(1).to_string())

Both combinations are **worse than the best individual model**. The mean comes in around 40 MAE behind the
random forest, and the median is not much better.

This is the caveat the reputation of ensembling tends to omit. An unweighted mean assumes the models are
roughly equally good. Ours are not: Ridge and SARIMAX are 70 to 80% worse than the random forest, and
averaging drags the good forecasts toward the bad ones. The median helps because it ignores the extremes,
but it still spends half its weight on models you would never ship.

**Averaging is only free when the models are comparable.** If one model is clearly better, an unweighted
mean is a way of discarding that knowledge.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="4.-Optimal-Weights">4. Optimal Weights</h3>
</div>

The fix is to weight the models. We want weights that are non-negative and sum to one, so the result
stays a proper average, and that minimise the error on data the models did not train on.

That is a constrained optimisation problem, and `scipy.optimize.minimize` solves it directly. Minimising
MAE rather than squared error keeps the objective consistent with how we have judged everything else.

In [ ]:
def optimal_weights(errors):
    """Non-negative weights summing to one that minimise the mean absolute error."""
    n_models = errors.shape[1]

    def objective(weights):
        return np.mean(np.abs(errors @ weights))

    result = minimize(
        objective,
        x0=np.ones(n_models) / n_models,
        bounds=[(0.0, 1.0)] * n_models,
        constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1.0}],
    )
    return pd.Series(result.x, index=errors.columns)


weights = optimal_weights(validation_errors)
combinations["Optimal weights"] = test_predictions.values @ weights.values

print("Weights learned on the validation period:")
print(weights.round(3).to_string())
print()
print(f"Optimal weights MAE: "
      f"{mean_absolute_error(y_test, combinations['Optimal weights']):.1f}")
print(f"Best individual:     {individual.iloc[0]:.1f}  ({individual.index[0]})")

The optimiser puts **zero weight on Ridge and SARIMAX**, splits the rest between the random forest and
LightGBM, and lands marginally ahead of the random forest alone.

Read that honestly: the weighted ensemble did not discover a clever combination. It discovered that two of
the four models are not worth including, and then mostly picked the best one. That is a perfectly good
outcome, and it is much closer to automated model selection than to the variance reduction that
combination is usually sold on.

Note also that zero weights are a real answer, not a failure. A method that can exclude a model is more
useful than one that cannot.

**Exercise.** Refit the weights with `Ridge` and `SARIMAX` removed from the pool entirely. Do the weights on the two tree models change, and does the test score move? What does that tell you about how much the zero-weighted models were contributing?

In [ ]:
# Your solution here


---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="5.-Stacking">5. Stacking</h3>
</div>

**Stacking** goes further. Instead of a weighted average, train a second model, the **meta-model**, that
takes the base forecasts as its features and learns how to map them onto the truth.

That lets the combination do things an average cannot: correct a systematic bias, or lean on one model in
some conditions and another elsewhere. The risk is equally clear. The meta-model has its own capacity to
overfit, on a validation set that is usually small, so a simple meta-model is almost always the right
choice. A linear regression constrained to non-negative coefficients is a good default: it is a weighted
average that is also allowed to shift the level.

**Blending** is the same idea with a single holdout rather than cross-validation, which is exactly the
setup we are using here.

In [ ]:
meta_model = LinearRegression(positive=True)
meta_model.fit(validation_predictions, y_validation)

combinations["Stacking"] = meta_model.predict(test_predictions)

print("Meta-model coefficients:")
print(pd.Series(meta_model.coef_, index=validation_predictions.columns).round(3).to_string())
print(f"intercept: {meta_model.intercept_:.1f}")
print()
print(f"Stacking MAE: {mean_absolute_error(y_test, combinations['Stacking']):.1f}")

In [ ]:
all_scores = pd.concat([
    individual.rename("MAE").to_frame().assign(Kind="Individual model"),
    pd.Series(
        {name: mean_absolute_error(y_test, combinations[name]) for name in combinations},
        name="MAE",
    ).to_frame().assign(Kind="Combination"),
]).sort_values("MAE")

fig, ax = plt.subplots(figsize=(10, 5))

palette = {"Individual model": "steelblue", "Combination": "seagreen"}
ordered = all_scores.iloc[::-1]
ax.barh(ordered.index, ordered["MAE"], color=[palette[k] for k in ordered["Kind"]])

for position, value in enumerate(ordered["MAE"]):
    ax.text(value + 5, position, f"{value:.0f}", va="center", fontsize=9)

best_line = ax.axvline(individual.iloc[0], color="crimson", linestyle="--", linewidth=1.2)

handles = [plt.Rectangle((0, 0), 1, 1, color=colour) for colour in palette.values()]
labels = list(palette)
ax.legend(handles + [best_line], labels + [f"Best individual ({individual.iloc[0]:.0f})"],
          loc="lower right")
ax.set_title("One test period", fontsize=13, fontweight="bold")
ax.set_xlabel("MAE")
ax.set_xlim(0, ordered["MAE"].max() * 1.15)
ax.grid(axis="x", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

On this single test period the ranking is close and the differences are small. Which is precisely the
situation Notebook [A06](./A06_Evaluating_models.ipynb) warned against reading anything into.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="6.-Does-It-Actually-Help?">6. Does It Actually Help?</h3>
</div>

One test period cannot settle this. The claim made for ensembles is usually about **robustness**: not that
the combination wins on any given period, but that it avoids the bad periods a single model occasionally
has.

That is a claim about the distribution of results across many origins, so we measure it the way A06 did.
Six origins, sixty days each, everything refitted from scratch at every one.

> **This cell takes a couple of minutes.** It fits four models, twice, at six origins.

In [ ]:
def evaluate_at_origin(X, y, exogenous, test_start, horizon, validation_days=90):
    """Score every model and every combination at one forecast origin."""
    validation_start = test_start - validation_days

    validation_predictions, test_predictions = fit_base_models(
        X, y, exogenous, validation_start, test_start, horizon
    )

    y_validation = y.iloc[validation_start:test_start]
    y_test = y.iloc[test_start:test_start + horizon]

    scores = {
        name: mean_absolute_error(y_test, test_predictions[name])
        for name in test_predictions
    }

    errors = validation_predictions.sub(y_validation, axis=0)
    weights = optimal_weights(errors)

    meta_model = LinearRegression(positive=True).fit(validation_predictions, y_validation)

    scores["Mean"] = mean_absolute_error(y_test, test_predictions.mean(axis=1))
    scores["Median"] = mean_absolute_error(y_test, test_predictions.median(axis=1))
    scores["Optimal weights"] = mean_absolute_error(y_test, test_predictions.values @ weights.values)
    scores["Stacking"] = mean_absolute_error(y_test, meta_model.predict(test_predictions))

    return scores


HORIZON = 60
N_ORIGINS = 6

by_origin = pd.DataFrame([
    evaluate_at_origin(X, y, exogenous, len(X) - HORIZON - (N_ORIGINS - 1 - k) * HORIZON, HORIZON)
    for k in range(N_ORIGINS)
])
by_origin.index = [f"origin {k + 1}" for k in range(N_ORIGINS)]

by_origin.round(0)

In [ ]:
summary = by_origin.agg(["mean", "std", "max"]).T.sort_values("mean")
summary.columns = ["Mean MAE", "Std across origins", "Worst origin"]

summary.round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

order = summary.index.tolist()
melted = by_origin[order].melt(var_name="Method", value_name="MAE")

sns.boxplot(data=melted, x="Method", y="MAE", ax=ax, width=0.55, color="lightsteelblue")
sns.stripplot(data=melted, x="Method", y="MAE", ax=ax, color="black", size=4, alpha=0.7)

ax.set_title(f"MAE across {N_ORIGINS} forecast origins", fontsize=13, fontweight="bold")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=30)
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

Now the picture is clear enough to draw a conclusion, and it is a measured one.

**Optimal weighting matches the best single model and is slightly steadier.** 358.3 against the random
forest's 358.6 on average, with a marginally smaller spread across origins. After all that machinery, the
accuracy gain is essentially zero.

**Stacking is a little worse than both**, at 364.6. Its extra flexibility bought nothing on 90 validation
days, which is the overfitting risk described in section 5 showing up as expected.

**The unweighted mean is clearly worse than either tree model**, at 408.3, and the median sits between
them. Both are dragged down by the two weak models, at every origin.

So does ensembling help? On this problem, **not in accuracy**. What it does provide is worth naming
precisely: the weighted ensemble reached the performance of the best model *without anyone having to know
in advance which model that would be*. At origin 1, LightGBM beat the random forest; at the other five, the
forest won. Committing to a single model means being right about that choice, and being right again when
the data shifts. The ensemble is insurance against that decision, bought at the cost of running four
models instead of one.

Whether that insurance is worth its premium depends on how confident you are, and how expensive it is to
be wrong.

**Exercise.** Drop the random forest from the pool and rerun the comparison across origins. Does the ensemble now beat the best remaining individual model? What does the answer suggest about when combination is most valuable?

In [ ]:
# Your solution here


---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="7.-When-to-Combine">7. When to Combine</h3>
</div>

| Method | Fitting needed | Use it when |
|---|---|---|
| **Mean** | None | Models are of comparable quality |
| **Median** | None | One model occasionally misbehaves |
| **Optimal weights** | A validation period | Models differ in quality, as here |
| **Stacking** | A validation period | Many validation points, and a reason to expect conditional behaviour |

What actually determines whether combining pays:

**Comparable quality.** An unweighted average of a good model and a bad one is worse than the good one.
If your models differ, weight them or drop the weak ones.

**Uncorrelated errors.** Our models correlated at 0.69 to 0.93 because they shared a feature matrix. Real
diversity comes from different data or genuinely different assumptions, not from a second tree library.

**Enough validation data to learn the combination.** Weights and meta-models are themselves fitted, on a
period that is usually much smaller than the training set.

**A reason beyond accuracy.** Ensembles buy insurance against picking the wrong model, and stability when
the best model changes over time. Those are real benefits that a single accuracy number does not show.

The honest summary for this dataset: a well-weighted ensemble matched the best single model and never
embarrassed itself. That is a good outcome, and it is not the dramatic improvement the technique is often
credited with.

---

That completes Part C. Forecasting as supervised learning, the models that go on top, and the ways of
combining them, all measured against the statistical methods of Part B and the baselines of Part A.

Part D turns to deep learning, where the models build their own features from raw sequences rather than
being handed a matrix, and where the questions of evaluation, leakage and baselines stay exactly as they
are here.

**Solutions.** Worked answers to the 2 exercises above, with the reasoning behind them, are in
[C03_Ensembles_solutions.ipynb](../solutions/C03_Ensembles_solutions.ipynb). Try each one yourself first.
